# CEF 2026 — FERLD cartographic reports in Colab

This notebook creates publication-ready PDF/PNG map reports directly in Google Colab, without QGIS and without local computer paths. It reads the outputs generated by the main FERLD forest-loss notebook from `My Drive/Workshop/Outputs`.

## 1. Install libraries

Colab is temporary, so these packages are installed at the beginning of each session.

In [ ]:
!pip install -q rasterio geopandas matplotlib contextily shapely pyproj

## 2. Mount Google Drive and locate workshop files

Expected structure:

```text
My Drive/
└── Workshop/
    ├── Limits/
    │   └── FERLD.geojson
    └── Outputs/
        ├── FERLD_RF_forest_loss_no_loss.tif
        ├── FERLD_temporal_NDVI_2017.tif
        ├── FERLD_temporal_NDVI_2020.tif
        ├── FERLD_temporal_NDVI_2023.tif
        └── FERLD_temporal_change_map_2017_2023.tif
```

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

WORKDIR = Path('/content/drive/MyDrive/Workshop')
LIMITS_DIR = WORKDIR / 'Limits'
OUTPUT_DIR = WORKDIR / 'Outputs'
REPORT_DIR = OUTPUT_DIR / 'Reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

FERLD_PATH = LIMITS_DIR / 'FERLD.geojson'
RF_TIF = OUTPUT_DIR / 'FERLD_RF_forest_loss_no_loss.tif'
NDVI_2017 = OUTPUT_DIR / 'FERLD_temporal_NDVI_2017.tif'
NDVI_2020 = OUTPUT_DIR / 'FERLD_temporal_NDVI_2020.tif'
NDVI_2023 = OUTPUT_DIR / 'FERLD_temporal_NDVI_2023.tif'
CHANGE_TIF = OUTPUT_DIR / 'FERLD_temporal_change_map_2017_2023.tif'

required = [FERLD_PATH, RF_TIF, NDVI_2017, NDVI_2020, NDVI_2023, CHANGE_TIF]
missing = [p for p in required if not p.exists()]

print('WORKDIR   :', WORKDIR)
print('OUTPUTS   :', OUTPUT_DIR)
print('REPORTS   :', REPORT_DIR)
print('\nFiles:')
for p in required:
    print(('OK      ' if p.exists() else 'MISSING '), p.name)

if missing:
    raise FileNotFoundError('Missing required files: ' + ', '.join(str(p) for p in missing))

## 3. Cartographic helper functions

The report uses high-contrast colors for change detection: magenta for loss, bright green for gain, and a very light stable class so the satellite basemap remains readable.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import rasterio
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.colors import ListedColormap, BoundaryNorm, LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from rasterio.plot import plotting_extent

try:
    import contextily as ctx
    HAS_BASEMAP = True
except Exception:
    HAS_BASEMAP = False

INK = '#1f2933'
MUTED = '#5b6770'
LOSS = '#e6007e'
GAIN = '#00b894'
STABLE = '#f7f7f7'
FERLD = '#ffd43b'
BUFFER = '#42c5f5'

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 12,
    'axes.labelsize': 9,
    'figure.facecolor': 'white',
})

def read_raster(path):
    src = rasterio.open(path)
    arr = src.read(1, masked=True)
    extent = plotting_extent(src)
    return src, arr, extent

def add_basemap(ax, crs):
    if not HAS_BASEMAP:
        ax.set_facecolor('#eef2f3')
        return
    try:
        ctx.add_basemap(
            ax,
            crs=crs,
            source=ctx.providers.Esri.WorldImagery,
            attribution=False,
            alpha=0.95,
        )
    except Exception as exc:
        ax.set_facecolor('#eef2f3')
        print('Basemap unavailable:', exc)

def workshop_geometries(target_crs):
    ferld_raw = gpd.read_file(FERLD_PATH).to_crs('EPSG:32618')
    try:
        ferld_geom = ferld_raw.geometry.union_all()
    except AttributeError:
        ferld_geom = ferld_raw.geometry.unary_union
    ferld = gpd.GeoDataFrame(geometry=[ferld_geom], crs='EPSG:32618').to_crs(target_crs)
    buffer = gpd.GeoDataFrame(geometry=[ferld_geom.buffer(10000)], crs='EPSG:32618').to_crs(target_crs)
    return ferld, buffer

def add_boundaries(ax, crs, show_buffer=False):
    ferld, buffer = workshop_geometries(crs)
    if show_buffer:
        buffer.boundary.plot(ax=ax, color=BUFFER, linewidth=0.9, alpha=0.55, zorder=7)
    ferld.boundary.plot(ax=ax, color=FERLD, linewidth=1.8, zorder=8)

def set_focus_extent(ax, crs, pad=0.45):
    ferld, _ = workshop_geometries(crs)
    xmin, ymin, xmax, ymax = ferld.total_bounds
    dx = xmax - xmin
    dy = ymax - ymin
    ax.set_xlim(xmin - dx * pad, xmax + dx * pad)
    ax.set_ylim(ymin - dy * pad, ymax + dy * pad)

def add_scale_bar(ax, length_km=2):
    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()
    width = xmax - xmin
    height = ymax - ymin
    x0 = xmin + width * 0.07
    y0 = ymin + height * 0.07
    length = length_km * 1000
    ax.plot([x0, x0 + length], [y0, y0], color='white', linewidth=5, solid_capstyle='butt', zorder=20)
    ax.plot([x0, x0 + length], [y0, y0], color=INK, linewidth=2, solid_capstyle='butt', zorder=21)
    ax.text(
        x0 + length / 2,
        y0 + height * 0.018,
        f'{length_km} km',
        ha='center', va='bottom', fontsize=8, color=INK,
        path_effects=[pe.withStroke(linewidth=3, foreground='white')],
        zorder=22,
    )

def clean_axis(ax):
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_linewidth(0.8)
        spine.set_edgecolor('#d8dee4')

def save_figure(fig, stem):
    png = REPORT_DIR / f'{stem}.png'
    pdf = REPORT_DIR / f'{stem}.pdf'
    fig.savefig(png, dpi=220, bbox_inches='tight')
    fig.savefig(pdf, dpi=220, bbox_inches='tight')
    print('Saved:', png)
    print('Saved:', pdf)

## 4. Main RF forest-loss map

The class `0` is rendered nearly transparent. The predicted forest-loss class is rendered in magenta so it does not disappear over the satellite background.

In [ ]:
src, rf, extent = read_raster(RF_TIF)
crs = src.crs

rf_cmap = ListedColormap([
    (247/255, 247/255, 247/255, 0.08),
    (230/255, 0/255, 126/255, 0.92),
])
rf_norm = BoundaryNorm([-0.5, 0.5, 1.5], rf_cmap.N)

fig, ax = plt.subplots(figsize=(13.5, 9.2))

set_focus_extent(ax, crs, pad=0.65)
add_basemap(ax, crs)
ax.imshow(rf, extent=extent, cmap=rf_cmap, norm=rf_norm, interpolation='nearest', zorder=5)
add_boundaries(ax, crs)
add_scale_bar(ax, 2)
clean_axis(ax)
ax.set_title('Perte de couvert forestier predite — FERLD', loc='left', fontweight='bold', color=INK, pad=10, fontsize=15)
ax.text(0.0, 1.015, 'Random Forest · Sentinel-2 SR · Hansen GFC v1.11 · 2001-2023', transform=ax.transAxes, color=MUTED, fontsize=10)

legend_items = [
    Patch(facecolor=LOSS, edgecolor='none', label='Perte de foret predite RF'),
    Line2D([0], [0], color=FERLD, lw=2, label='Limite FERLD'),
]
ax.legend(handles=legend_items, loc='lower right', frameon=True, framealpha=0.92, facecolor='white', edgecolor='#d8dee4', fontsize=9)
fig.text(0.08, 0.035, 'Random Forest: 300 arbres · 8 predicteurs spectraux · 400 points par classe · precision globale ~74 %', color=MUTED, fontsize=9)
fig.text(0.08, 0.015, 'Sentinel-2 SR (ESA), 2023 · Hansen Global Forest Change v1.11 · CEF Workshop 2026 · GEE + Python', color=MUTED, fontsize=8)

save_figure(fig, 'CEF2026_carte_FERLD_colab')
plt.show()
src.close()

## 5. Temporal NDVI and classified change map

The bottom-right panel uses the classified change map instead of continuous Delta NDVI. This makes real change visible at landscape scale.

In [ ]:
ndvi_cmap = LinearSegmentedColormap.from_list('ndvi', ['#f7f7f7', '#d9f0a3', '#78c679', '#006837'])
change_cmap = ListedColormap([
    (247/255, 247/255, 247/255, 0.20),
    (230/255, 0/255, 126/255, 0.94),
    (0/255, 184/255, 148/255, 0.94),
])
change_norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5], change_cmap.N)

rasters = [
    (NDVI_2017, 'NDVI 2017', ndvi_cmap, None, 0, 1),
    (NDVI_2020, 'NDVI 2020', ndvi_cmap, None, 0, 1),
    (NDVI_2023, 'NDVI 2023', ndvi_cmap, None, 0, 1),
    (CHANGE_TIF, 'Changement 2017-2023', change_cmap, change_norm, None, None),
]

fig, axes = plt.subplots(2, 2, figsize=(16.5, 11.7), constrained_layout=True)
axes = axes.ravel()

for ax, (path, title, cmap, norm, vmin, vmax) in zip(axes, rasters):
    src, arr, extent = read_raster(path)
    crs = src.crs
    set_focus_extent(ax, crs, pad=0.65)
    add_basemap(ax, crs)
    if norm is None:
        im = ax.imshow(arr, extent=extent, cmap=cmap, vmin=vmin, vmax=vmax, interpolation='nearest', alpha=0.88, zorder=5)
    else:
        im = ax.imshow(arr, extent=extent, cmap=cmap, norm=norm, interpolation='nearest', zorder=5)
    add_boundaries(ax, crs)
    clean_axis(ax)
    ax.set_title(title, loc='left', fontweight='bold', color=INK)
    src.close()

add_scale_bar(axes[2], 2)
fig.suptitle('Evolution temporelle du NDVI — FERLD', fontsize=18, fontweight='bold', color=INK)
fig.text(0.5, 0.015, 'Changement classe: perte Delta NDVI < -0.10 · gain > +0.05 · Sentinel-2 SR · CEF 2026', ha='center', color=MUTED, fontsize=10)

legend_items = [
    Patch(facecolor=LOSS, edgecolor='none', label='Perte Delta NDVI'),
    Patch(facecolor=GAIN, edgecolor='none', label='Gain Delta NDVI'),
    Line2D([0], [0], color=FERLD, lw=2, label='Limite FERLD'),
]
axes[3].legend(handles=legend_items, loc='lower left', frameon=True, framealpha=0.92, fontsize=9)

save_figure(fig, 'CEF2026_carte_temporelle_NDVI_colab')
plt.show()

## 6. Statistical summary PDF

This table is computed directly from the NDVI rasters using the same thresholds as the map.

In [ ]:
def change_stats(path_a, path_b, loss_thr=-0.10, gain_thr=0.05):
    with rasterio.open(path_a) as a, rasterio.open(path_b) as b:
        arr_a = a.read(1, masked=True).astype('float32')
        arr_b = b.read(1, masked=True).astype('float32')
        rows = min(arr_a.shape[0], arr_b.shape[0])
        cols = min(arr_a.shape[1], arr_b.shape[1])
        arr_a = arr_a[:rows, :cols]
        arr_b = arr_b[:rows, :cols]
        valid = (~arr_a.mask) & (~arr_b.mask) & (arr_a > -9999) & (arr_b > -9999)
        delta = arr_b - arr_a
        stable = int(np.sum((delta >= loss_thr) & (delta <= gain_thr) & valid))
        loss = int(np.sum((delta < loss_thr) & valid))
        gain = int(np.sum((delta > gain_thr) & valid))
        pixel_area_ha = abs(a.transform.a * a.transform.e) / 10000
        total = stable + loss + gain
        return {
            'Stable (ha)': round(stable * pixel_area_ha),
            'Perte (ha)': round(loss * pixel_area_ha),
            'Gain (ha)': round(gain * pixel_area_ha),
            'Total (ha)': round(total * pixel_area_ha),
            '% Perte': round(loss / total * 100, 1) if total else 0,
            '% Gain': round(gain / total * 100, 1) if total else 0,
        }

stats = {
    '2017-2020': change_stats(NDVI_2017, NDVI_2020),
    '2020-2023': change_stats(NDVI_2020, NDVI_2023),
    '2017-2023': change_stats(NDVI_2017, NDVI_2023),
}

import pandas as pd
df = pd.DataFrame(stats).T
display(df)

fig, ax = plt.subplots(figsize=(11.7, 8.3))
ax.axis('off')
ax.text(0.03, 0.94, 'Analyse quantitative des changements de couvert vegetal', fontsize=18, fontweight='bold', color=INK, transform=ax.transAxes)
ax.text(0.03, 0.89, 'FERLD + zone tampon 10 km · Sentinel-2 SR · 2017 / 2020 / 2023', fontsize=11, color=MUTED, transform=ax.transAxes)

table = ax.table(
    cellText=df.reset_index().values,
    colLabels=['Periode'] + list(df.columns),
    loc='center',
    cellLoc='center',
    colLoc='center',
    bbox=[0.03, 0.48, 0.94, 0.28],
)
table.auto_set_font_size(False)
table.set_fontsize(9)
for (row, col), cell in table.get_celld().items():
    cell.set_edgecolor('#d8dee4')
    if row == 0:
        cell.set_facecolor('#1f2933')
        cell.get_text().set_color('white')
        cell.get_text().set_fontweight('bold')
    elif col == 0:
        cell.get_text().set_fontweight('bold')
        cell.set_facecolor('#f7f7f7')

ax.text(0.03, 0.36, 'Interpretation', fontsize=13, fontweight='bold', color=INK, transform=ax.transAxes)
ax.text(
    0.03, 0.23,
    'Stable: Delta NDVI between -0.10 and +0.05.\n'
    'Perte: Delta NDVI < -0.10, indicating loss of vegetation vigor.\n'
    'Gain: Delta NDVI > +0.05, indicating vegetation recovery or canopy closure.\n'
    'Delta NDVI detects spectral change; causal interpretation should be crossed with Hansen GFC and field knowledge.',
    fontsize=10, color=MUTED, linespacing=1.6, transform=ax.transAxes,
)
ax.text(0.03, 0.06, 'CEF Workshop 2026 · Google Earth Engine + Python · Report generated in Google Colab', fontsize=9, color=MUTED, transform=ax.transAxes)

save_figure(fig, 'CEF2026_tableau_statistique_NDVI_colab')
plt.show()

## Outputs

The generated reports are saved in:

`My Drive/Workshop/Outputs/Reports/`

Files generated:

- `CEF2026_carte_FERLD_colab.pdf/png`
- `CEF2026_carte_temporelle_NDVI_colab.pdf/png`
- `CEF2026_tableau_statistique_NDVI_colab.pdf/png`